In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

#                                             Counter : Strike Physics Engine

#### Reason to elaborate on this topic 

In my free time (when there is a possibility to have one), I really enjoy playing video games, especially Counter-Strike. When I saw that there is a chance I can create a (2D, of course) shooting engine to understand a bit further how the damage is calculated in the game, I took the opportunity to do so.

### Plan: 
1. Weapon - characteristics
2. Introduction to Hitscan - shooting representation
3. What happens when the bullet trajectory hits a wall
4. What happens when the bullet goes through a wallbangable wall? 
5. Utility - grenades, molotov - Newton 

## Weapon 
For the purpose of this project, I will use AK-47 (Kalashnikov) - one of the most known Counter:Strike weapon. It will have damage, bullets and recoil.


The weapon has three properties that change during gameplay:
- **bullets** - starts at 30, decreases by 1 with each shot
- **damage** - stays at 36 HP per hit (simplified)
- **recoil** - starts at 0 and increases by 7 every time you fire

Mathematically, if we fire $n$ shots:

$$
\text{bullets}(n) = 30 - n, \qquad \text{recoil}(n) = 7n
$$

The weapon's state after $n$ shots is just the vector $\mathbf{W}(n) = (30 - n,\; 36,\; 7n)$.


In [ ]:
class AK47:
    def __init__(self):
        self.bullets = 30
        self.damage = 36
        self.recoil = 0

    def shoot(self):
        if self.bullets > 0:
            self.bullets -= 1
            self.recoil += 7
            return True
        return False

ak = AK47()
print(f"Fresh AK-47: bullets={ak.bullets}, damage={ak.damage}, recoil={ak.recoil}")

## 2. Introduction to Hitscan - shooting representation

In Counter-Strike, bullets do not actually fly through the air. The moment you click, the game draws an invisible line from your position in the direction you are aiming, checks what that line hits, and applies damage instantly. This is called **hitscan**.

Formally, the bullet is a ray starting at the shooter's position $\vec{P}$ going in direction $\hat{D}$:

$$
\vec{R}(t) = \vec{P} + t\,\hat{D}, \qquad t \geq 0
$$

where $t$ is the distance along the ray. Everything here is 2D, so a position is just a tuple $(x, y)$.

We need a few vector operations to make this work:

**Norm** (length of a vector): $\lVert \vec{v} \rVert = \sqrt{v_x^2 + v_y^2}$

**Normalize** (make it length 1): $\hat{v} = \vec{v}\,/\,\lVert \vec{v} \rVert$

**Rotation** by angle $\theta$:

$$
R(\theta)\,\vec{v} = \begin{pmatrix} v_x\cos\theta - v_y\sin\theta \\ v_x\sin\theta + v_y\cos\theta \end{pmatrix}
$$

And the big one - **ray-wall intersection**. A wall is a line segment from point $\vec{A}$ to point $\vec{B}$, parameterized as $\vec{W}(s) = \vec{A} + s(\vec{B} - \vec{A})$ for $s \in [0,1]$. To find where the ray hits the wall, we solve:

$$
\vec{P} + t\,\hat{D} = \vec{A} + s\,\vec{v}
$$

where $\vec{v} = \vec{B} - \vec{A}$. Using Cramer's rule:

$$
t = \frac{(A_x - P_x)\,v_y - (A_y - P_y)\,v_x}{\hat{D}_x\,v_y - \hat{D}_y\,v_x}
$$

If $t \geq 0$ and $s \in [0,1]$, the bullet hits the wall at point $\vec{H} = \vec{P} + t\,\hat{D}$.

In [ ]:
def norm(v):
    return math.sqrt(v[0]**2 + v[1]**2)

def normalize(v):
    m = norm(v)
    return (v[0]/m, v[1]/m) if m > 0 else (0.0, 0.0)

def rotate(v, angle_deg):
    r = math.radians(angle_deg)
    c, s = math.cos(r), math.sin(r)
    return (v[0]*c - v[1]*s, v[0]*s + v[1]*c)

def ray_wall_intersection(origin, direction, wall_start, wall_end):
    """Returns (hit, t, point) using Cramer's rule."""
    d = normalize(direction)
    vx = wall_end[0] - wall_start[0]
    vy = wall_end[1] - wall_start[1]

    det = d[0]*vy - d[1]*vx
    if abs(det) < 1e-10:
        return (False, None, None)

    dx = wall_start[0] - origin[0]
    dy = wall_start[1] - origin[1]

    t = (dx*vy - dy*vx) / det
    s = (dx*d[1] - dy*d[0]) / det

    if t >= 0 and 0 <= s <= 1:
        hit = (origin[0] + d[0]*t, origin[1] + d[1]*t)
        return (True, t, hit)

    return (False, None, None)

# quick sanity check: shoot straight right at a vertical wall
hit, t, point = ray_wall_intersection((0, 5), (1, 0), (10, 0), (10, 10))
print(f"Hit: {hit}, distance: {t}, point: ({point[0]:.1f}, {point[1]:.1f})")

Create test figures